
# product_info (1).json → product_info_preprocessed.json/.jsonl

최신 `product_info (1).json`을 **프롬프트 친화 형태**로 전처리하는 노트북입니다.

- 입력: `/mnt/data/product_info (1).json`
- 출력: `/mnt/data/product_info_preprocessed.json`, `/mnt/data/product_info_preprocessed.jsonl`
- 규칙:
  - `feature`, `targeted_consumer` → 리스트 분해(쉼표/세미콜론/슬래시 기준)
  - `category.level_1~3` → `"레벨1 > 레벨2 > 레벨3"`
  - `advertise{date, how}` → `"광고/프로모션: {date}, {how}"`
  - `price` → `"#,###원"` 포맷
  - `release_date` → `"YYYY년 M월 출시"` (없으면 미표기)
  - `ad_model` 값이 없으면 미표기
  - `prompt_block` 생성 (프롬프트의 [제품 정보]에 그대로 삽입 용)


In [7]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

INPUT_JSON  = Path("product_info.json")
OUT_JSON    = Path("product_info_preprocessed.json")
OUT_JSONL   = Path("product_info_preprocessed.jsonl")
brand = "동원 F&B"

# (선택) placeholder 사용 여부: 값이 없을 때도 필드 표시
USE_PLACEHOLDERS = False
PLACEHOLDERS = {
    "release_info": "출시일: 정보 없음",
    "ad_model": "광고모델: 없음"
}

print("CONFIG loaded.")

CONFIG loaded.


In [8]:

# =============================
# 1) Helpers
# =============================
import json, re

def split_list(val):
    if val is None:
        return []
    parts = re.split(r"[;,/]", str(val))
    return [p.strip() for p in parts if p and p.strip().lower() != "nan"]

def to_price_text(v):
    if v is None:
        return None
    try:
        iv = int(round(float(v)))
        return f"{iv:,}원"
    except Exception:
        return str(v)

def to_release_info(v):
    if v is None:
        return None
    s = str(v).strip()
    if not s:
        return None
    s = re.sub(r"[./]", "-", s)
    m = re.match(r"^(\d{4})-(\d{1,2})(?:-(\d{1,2}))?$", s)
    if m:
        y, mo = int(m.group(1)), int(m.group(2))
        return f"{y}년 {mo}월 출시"
    if "년" in s:
        return s if "출시" in s else f"{s} 출시"
    return f"{s} 출시"

def to_category_str(cat):
    if not isinstance(cat, dict):
        return None
    parts = [cat.get("level_1"), cat.get("level_2"), cat.get("level_3")]
    parts = [p for p in parts if p]
    return " > ".join(parts) if parts else None

def to_advertise_info(ad):
    if ad is None:
        return None
    if isinstance(ad, dict):
        date = (ad.get("date") or "").strip(" ,")
        how  = (ad.get("how") or "").strip(" ,")
        pieces = [p for p in [date, how] if p]
        return ("광고/프로모션: " + ", ".join(pieces)) if pieces else None
    return f"광고/프로모션: {ad}"

def preprocess_item(r, use_placeholders=False):
    product_id = r.get("product_id") or r.get("id")
    name = r.get("product_name")
    category_str = to_category_str(r.get("category"))
    features = split_list(r.get("feature") or r.get("features"))
    targeted = split_list(r.get("targeted_consumer"))
    release_info = to_release_info(r.get("release_date") or r.get("launch_ym"))
    price_text = to_price_text(r.get("price"))
    advertise_info = to_advertise_info(r.get("advertise"))
    ad_model = r.get("ad_model")

    out = {
        "product_id": product_id,
        "brand": brand,
        "product_name": name,
        "category": category_str,
        "features": features,
        "targeted_consumer": targeted,
        "release_info": release_info,
        "price_text": price_text,
        "advertise_info": advertise_info,
        "ad_model": ad_model
    }

    # Build prompt_block (빈 값은 기본적으로 생략)
    bullets = []
    if name: bullets.append(f"- 제품명: {name}")
    if category_str: bullets.append(f"- 카테고리: {category_str}")
    if features: bullets.append(f"- 주요 특징: {', '.join(features)}")
    if targeted: bullets.append(f"- 타깃: {', '.join(targeted)}")
    if release_info: bullets.append(f"- 출시일: {release_info}")
    if price_text: bullets.append(f"- 기준 가격대: {price_text}")
    if ad_model: bullets.append(f"- 광고모델: {ad_model}")
    if advertise_info: bullets.append(f"- {advertise_info}")

    if use_placeholders:
        if not release_info:
            bullets.append(f"- {PLACEHOLDERS['release_info']}")
        if not ad_model:
            bullets.append(f"- {PLACEHOLDERS['ad_model']}")

    out["prompt_block"] = "\n".join(bullets)
    return out


In [9]:

# =============================
# 2) Load → Preprocess → Save
# =============================
raw = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
records = raw if isinstance(raw, list) else [raw]

preprocessed = [preprocess_item(r, use_placeholders=USE_PLACEHOLDERS) for r in records]

# Save JSON
OUT_JSON.write_text(json.dumps(preprocessed, ensure_ascii=False, indent=2), encoding="utf-8")

# Save JSONL
with OUT_JSONL.open("w", encoding="utf-8") as f:
    for rec in preprocessed:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Saved ->", OUT_JSON)
print("Saved ->", OUT_JSONL)
len(preprocessed), preprocessed[:2]

Saved -> product_info_preprocessed.json
Saved -> product_info_preprocessed.jsonl


(15,
 [{'product_id': None,
   'brand': '동원 F&B',
   'product_name': '덴마크 하이그릭요거트 400g',
   'category': '우유류 > 발효유 > 호상-중대용량',
   'features': ['건강식품', '고단백', '고소한맛', '높은 만족도'],
   'targeted_consumer': ['유당불내증'],
   'release_info': '2025년 2월 출시',
   'price_text': '3,980원',
   'advertise_info': '광고/프로모션: 2025년 6-7월, 일반인 광고',
   'ad_model': None,
   'prompt_block': '- 제품명: 덴마크 하이그릭요거트 400g\n- 카테고리: 우유류 > 발효유 > 호상-중대용량\n- 주요 특징: 건강식품, 고단백, 고소한맛, 높은 만족도\n- 타깃: 유당불내증\n- 출시일: 2025년 2월 출시\n- 기준 가격대: 3,980원\n- 광고/프로모션: 2025년 6-7월, 일반인 광고'},
  {'product_id': None,
   'brand': '동원 F&B',
   'product_name': '동원맛참 고소참기름 135g',
   'category': '참치 > 참치캔 > 라이트스탠다드참치',
   'features': ['간편함', '고단백', '고소한맛', '가성비', '높은 만족도'],
   'targeted_consumer': ['1세대가족', '2세대가족'],
   'release_info': '2023년 8월 출시',
   'price_text': '2,500원',
   'advertise_info': '광고/프로모션: 2024년 5-12월, 연예인 광고',
   'ad_model': None,
   'prompt_block': '- 제품명: 동원맛참 고소참기름 135g\n- 카테고리: 참치 > 참치캔 > 라이트스탠다드참치\n- 주요 특징: 간편함, 고단백, 고소한맛, 가성비, 높